# Mini-Batch Gradient Descent

> Based on Stanford CS230 — C2M2, Andrew Ng / DeepLearning.AI

Instead of using all $m$ examples (batch GD) or one at a time (SGD), **mini-batch GD** splits the data into small batches of size $B$ and performs one parameter update per batch. This balances speed with stability.

## Learning Objectives

1. Explain the trade-offs between batch, mini-batch, and stochastic gradient descent
2. Implement a mini-batch training loop with random shuffling
3. Visualise how batch size affects the cost curve noise level
4. Understand exponentially weighted averages — the building block of momentum / Adam

## Batch Size Trade-offs

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 620 160" width="620" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Headers -->
  <text x="100" y="22" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Batch GD  (B = m)</text>
  <text x="310" y="22" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Mini-Batch GD  (B = 64…512)</text>
  <text x="520" y="22" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">SGD  (B = 1)</text>
  <!-- Dividers -->
  <line x1="200" y1="10" x2="200" y2="155" stroke="#e0e3ea" stroke-width="1" stroke-dasharray="4,3"/>
  <line x1="420" y1="10" x2="420" y2="155" stroke="#e0e3ea" stroke-width="1" stroke-dasharray="4,3"/>
  <!-- Batch GD column -->
  <text x="30"  y="45"  fill="#555" font-size="10">✓ Smooth cost curve</text>
  <text x="30"  y="62"  fill="#555" font-size="10">✓ Vectorised — fast per epoch</text>
  <text x="30"  y="79"  fill="#e05c5c" font-size="10">✗ Slow to start — waits for all m</text>
  <text x="30"  y="96"  fill="#e05c5c" font-size="10">✗ Memory heavy</text>
  <!-- Mini-batch column -->
  <text x="210" y="45"  fill="#555" font-size="10">✓ Best of both worlds</text>
  <text x="210" y="62"  fill="#555" font-size="10">✓ Vectorised per batch</text>
  <text x="210" y="79"  fill="#555" font-size="10">✓ Frequent updates</text>
  <text x="210" y="96"  fill="#7ecba1" font-size="11" font-weight="bold">← preferred in practice</text>
  <text x="210" y="113" fill="#555" font-size="10">Typical B: powers of 2 (64,128,256)</text>
  <!-- SGD column -->
  <text x="430" y="45"  fill="#555" font-size="10">✓ Very frequent updates</text>
  <text x="430" y="62"  fill="#e05c5c" font-size="10">✗ Very noisy cost curve</text>
  <text x="430" y="79"  fill="#e05c5c" font-size="10">✗ Loses vectorisation benefits</text>
  <text x="430" y="96"  fill="#555" font-size="10">Useful: online learning</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

def init_he(dims, seed=1):
    np.random.seed(seed)
    p = {}
    for l in range(1, len(dims)):
        p[f'W{l}'] = np.random.randn(dims[l], dims[l-1]) * np.sqrt(2/dims[l-1])
        p[f'b{l}'] = np.zeros((dims[l], 1))
    return p

def forward_backward(X_b, Y_b, params, L):
    m = Y_b.shape[1]
    # forward
    A, caches = X_b, []
    for l in range(1, L+1):
        W, b = params[f'W{l}'], params[f'b{l}']
        Z = W @ A + b
        A_p = A
        A = relu(Z) if l < L else sigmoid(Z)
        caches.append((A_p, W, b, Z, A))
    cost = -np.mean(Y_b*np.log(A+1e-8) + (1-Y_b)*np.log(1-A+1e-8))
    # backward
    grads = {}
    _, _, _, _, AL = caches[-1]
    dA = -(Y_b/(AL+1e-8) - (1-Y_b)/(1-AL+1e-8))
    for l in reversed(range(1, L+1)):
        A_p, W, b, Z, A_l = caches[l-1]
        dZ = dA * A_l*(1-A_l) if l==L else dA*(Z>0).astype(float)
        grads[f'dW{l}'] = (dZ @ A_p.T)/m
        grads[f'db{l}'] = np.mean(dZ, axis=1, keepdims=True)
        dA = W.T @ dZ
    return cost, grads

def make_batches(X, Y, batch_size, seed):
    np.random.seed(seed)
    m = X.shape[1]
    perm = np.random.permutation(m)
    X_s, Y_s = X[:, perm], Y[:, perm]
    batches = []
    for k in range(0, m, batch_size):
        batches.append((X_s[:, k:k+batch_size], Y_s[:, k:k+batch_size]))
    return batches

def train_minibatch(X, Y, dims, batch_size, lr=0.05, n_epochs=20, seed=1):
    L = len(dims)-1
    params = init_he(dims, seed=seed)
    costs = []
    for ep in range(n_epochs):
        batches = make_batches(X, Y, batch_size, seed=ep)
        for X_b, Y_b in batches:
            cost, grads = forward_backward(X_b, Y_b, params, L)
            costs.append(cost)
            for l in range(1, L+1):
                params[f'W{l}'] -= lr * grads[f'dW{l}']
                params[f'b{l}'] -= lr * grads[f'db{l}']
    return params, costs

# ---- Dataset ----
np.random.seed(0)
m = 1000
X = np.random.randn(4, m)
W_true = np.array([[1.5, -0.8, 0.3, 1.2]])
Y = (sigmoid(W_true @ X + 0.5) > 0.5).astype(float)
dims = [4, 16, 8, 1]

batch_configs = [
    (m,    'Batch GD (B=m)',       P[2]),
    (256,  'Mini-Batch B=256',     P[0]),
    (64,   'Mini-Batch B=64',      P[3]),
    (1,    'SGD (B=1)',            P[1]),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mini-Batch Gradient Descent — Batch Size Comparison', fontsize=13, fontweight='bold')

for batch_size, label, color in batch_configs:
    _, costs = train_minibatch(X, Y, dims, batch_size, lr=0.05, n_epochs=20)
    # plot vs update step
    axes[0].plot(costs, color=color, lw=1.5, alpha=0.85, label=f'{label}  ({len(costs)} steps)')

axes[0].set_xlabel('Parameter update step')
axes[0].set_ylabel('Cost per mini-batch')
axes[0].set_title('Cost Curve — Update Steps')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.5)
axes[0].grid(True)

# Per-epoch view: average cost per epoch
for batch_size, label, color in batch_configs:
    _, costs = train_minibatch(X, Y, dims, batch_size, lr=0.05, n_epochs=20)
    n_per_epoch = max(1, m // batch_size)
    epoch_costs = [np.mean(costs[i*n_per_epoch:(i+1)*n_per_epoch]) for i in range(20)]
    axes[1].plot(epoch_costs, color=color, lw=2.2, marker='o', ms=4, label=label)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Average cost per epoch')
axes[1].set_title('Cost Curve — Per Epoch')
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ---- Exponentially Weighted Averages (EWA) ----
# EWA is the core building block of momentum, RMSprop, and Adam.
# V_t = β * V_{t-1} + (1-β) * θ_t
# With bias correction: V_t_corrected = V_t / (1 - β^t)

np.random.seed(2)
n_days = 100
temperature = 70 + 10 * np.sin(np.linspace(0, 2*np.pi, n_days)) + np.random.randn(n_days) * 5

betas = [0.5, 0.9, 0.98]
colors = [P[2], P[0], P[3]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Exponentially Weighted Averages — Building Block of Momentum', fontsize=12, fontweight='bold')

for ax, bias_corr, title in zip(axes, [False, True], ['Without Bias Correction', 'With Bias Correction']):
    ax.plot(temperature, color='#ccc', lw=1.2, label='Raw data', zorder=1)
    for beta, color in zip(betas, colors):
        V = np.zeros(n_days)
        for t in range(n_days):
            V[t] = beta * (V[t-1] if t > 0 else 0) + (1-beta) * temperature[t]
        if bias_corr:
            V_plot = np.array([V[t] / (1 - beta**(t+1)) for t in range(n_days)])
        else:
            V_plot = V
        label = f'β={beta}  ≈ {int(1/(1-beta))} days avg'
        ax.plot(V_plot, color=color, lw=2, label=label, zorder=2)
    ax.set_xlabel('Day')
    ax.set_ylabel('Temperature (°F)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

print("EWA approximates average over last  1/(1−β)  time steps:")
for b in betas:
    print(f"  β={b}  →  ~{1/(1-b):.0f} steps")
